# 13 — Futures & Concurrency

Until now, every value you've worked with has been *here, now*. This notebook adds the time dimension. A `Future[A]` is a value that **will eventually** be an `A` — or a failure — but isn't yet. The work happens on another thread; your code keeps moving.

The good news is that you already know the shape. `Future` has `map`, `flatMap`, and `for`-comprehensions, exactly like `Option`, `Try`, and `Either` did in notebook 09. So composing async work feels like composing any other container. The new things to learn are runtime concerns:

- **`ExecutionContext`** — where the work actually runs.
- **Eagerness** — Futures start the instant you construct them. This is not laziness like `Option`.
- **Combinators for many futures** — `sequence`, `traverse`, `zip`, `firstCompletedOf`.
- **`Await`** — the blocking escape hatch you should almost never use.
- **A note on effect systems** — `cats-effect`'s `IO` and ZIO's `ZIO`, which solve problems `Future` doesn't.

## `Future[A]` — a value produced eventually

Conceptually, a `Future[A]` is a slot. The slot is empty when you first see it. Some computation, running on another thread, will eventually fill it with either a `Success(a)` of type `A` or a `Failure(throwable)`. Once filled, the slot never changes.

## `ExecutionContext` — where the work runs

Every `Future` operation needs an `ExecutionContext` — essentially a thread pool that knows how to run a block of code asynchronously. The compiler asks for it via a `using` parameter, so you don't write it at every call, but you must have one in scope.

The standard import gives you the global execution context, a sensibly configured pool that's the right default for CPU-bound work. For blocking I/O, you'd normally use a dedicated pool — more on that at the end.

In [ ]:
import scala.concurrent.{Future, ExecutionContext}
import scala.concurrent.ExecutionContext.Implicits.global
import scala.concurrent.duration.*

val f: Future[Int] = Future { 1 + 1 }
// f is now running (or already done) on the global EC. The current thread continues.

Two details to absorb from that small example.

- `Future { ... }` is **eager**. The block starts evaluating *as soon as you write that line*. Holding a `Future` in a `val` doesn't pause anything; the work is already on its way.
- The `import` of `global` is what supplies the `using ec: ExecutionContext` parameter that `Future.apply` requires. Without it the line wouldn't compile.

## Reading the result — callbacks via `onComplete`

The lowest-level way to react to a completed Future is to register a callback with `onComplete`. The callback receives a `Try[A]` — the same `Success`/`Failure` shape you met in notebook 09.

In [ ]:
import scala.util.{Success, Failure}

Future { 21 * 2 }.onComplete {
  case Success(n) => println(s"got $n")
  case Failure(e) => println(s"failed: ${e.getMessage}")
}
// prints "got 42" eventually, from whichever thread the EC chose

`onComplete` is useful at the *edges* of your program — logging a result, finishing an HTTP response. In the middle of a pipeline, callbacks lead to the same nesting problem you saw with `Option` before discovering `flatMap`. Reach for combinators instead.

## `map` and `flatMap` — transform without unwrapping

The same combinators you've seen on `Option`, `Try`, and `Either` are on `Future`. `map(f)` transforms the eventual value. `flatMap(f)` chains a second `Future` after the first. Each combinator is itself asynchronous — it returns a *new* `Future` without blocking the caller.

In [ ]:
def fetchUser(id: Long): Future[String]    = Future { Thread.sleep(50); s"user#$id" }
def fetchAccount(name: String): Future[Int] = Future { Thread.sleep(50); name.length * 100 }

val balance: Future[Int] =
  fetchUser(1).flatMap(name => fetchAccount(name))

// balance is itself a Future. Nothing has been waited on.

## `for` comprehensions — the readable form

Once you have two or more dependent steps, the `for` comprehension is the readable shape. Each `<-` is a `flatMap`, except the final one which is a `map` produced by `yield`. Familiar from notebook 09.

In [ ]:
val summary: Future[String] =
  for
    name    <- fetchUser(1)
    balance <- fetchAccount(name)
  yield s"$name owes $balance"

// Each step runs *after* the previous one completes — sequential.

## Sequential vs parallel — where you start the future matters

A `for` comprehension over Futures is **sequential by default**. Each step waits for the previous to complete because each one is created *inside* the body of the for. If you want parallelism, you start the futures *before* the for, then join their results.

In [ ]:
// Sequential — total time ~100ms (50 + 50)
val seqTotal: Future[Int] =
  for
    a <- Future { Thread.sleep(50); 1 }
    b <- Future { Thread.sleep(50); 2 }
  yield a + b

// Parallel — total time ~50ms
val fa = Future { Thread.sleep(50); 1 }   // starts immediately
val fb = Future { Thread.sleep(50); 2 }   // also starts immediately
val parTotal: Future[Int] =
  for
    a <- fa
    b <- fb
  yield a + b

Read those two carefully — the difference between them is the single most important Future idiom to internalise. In the sequential version, `fb`'s `Future` block is *constructed* only after `a` is bound, so it doesn't even start until then. In the parallel version, both blocks were already in flight before the `for` started.

The rule: **`for` doesn't parallelise — eagerness does.** Move the construction out of the for body to get overlap.

## Failure semantics — the first failure wins

If any step in a `for` comprehension fails, the whole chain short-circuits to that failure — exactly like `None` short-circuits an `Option` for. Subsequent steps don't run. The resulting `Future` is `Failure(throwable)`, and the original exception flows out unchanged.

In [ ]:
val broken: Future[Int] =
  for
    a <- Future { 10 }
    b <- Future { sys.error("boom") }      // throws — chain fails here
    c <- Future { 30 }                      // never runs
  yield a + b + c
// broken is a Failure(java.lang.RuntimeException: boom)

## `recover` and `recoverWith` — handling failures

Same names you saw on `Try`. `recover` turns a failure into a success with a plain value. `recoverWith` does the same but the recovery itself returns a `Future` — useful for retry-style logic.

In [ ]:
val safe: Future[Int] =
  Future { sys.error("down") }
    .recover { case _: RuntimeException => -1 }
// Future(Success(-1))

val retried: Future[Int] =
  Future { sys.error("flaky") }
    .recoverWith { case _: RuntimeException => Future { 42 } }
// Future(Success(42))

Two related combinators worth knowing. `transform` lets you handle success and failure in one place by mapping a `Try[A]` to a `Try[B]`. `transformWith` does the same but maps to a `Future[B]` — useful when one branch wants to chain another async step.

## Combining many futures — `sequence` and `traverse`

When you have a *collection* of futures and want one future of the collection, reach for `Future.sequence`. It turns a `List[Future[A]]` into a `Future[List[A]]`, completing when every member completes. If any one fails, the combined future fails immediately with the first failure.

In [ ]:
val ids = List(1L, 2L, 3L)

// Each Future.apply call starts immediately — these are in parallel.
val allUsers: Future[List[String]] =
  Future.sequence(ids.map(fetchUser))

When the pattern is *map each element through a `Future`-returning function, then sequence*, `Future.traverse` does both in one step and is more efficient because it avoids the intermediate list.

In [ ]:
val allUsersV2: Future[List[String]] =
  Future.traverse(ids)(fetchUser)

// Equivalent to the sequence version above, but no intermediate List[Future[String]].

## Pairing and racing — `zip` and `firstCompletedOf`

Two more small combinators round out the set.

- `f.zip(g)` returns a `Future[(A, B)]` that completes when both complete. Fails if either fails.
- `Future.firstCompletedOf(xs)` returns whichever of the supplied futures finishes first. Useful for timeouts and fallbacks.

In [ ]:
val both: Future[(String, Int)] =
  fetchUser(1).zip(fetchAccount("alice"))

val withTimeout: Future[Int] =
  Future.firstCompletedOf(Seq(
    fetchAccount("alice"),
    Future { Thread.sleep(10); 0 }   // a tiny timeout future
  ))

## Blocking — `Await`, the escape hatch

Sometimes you need to *force* the calling thread to wait for a `Future` to complete — the boundary between async code and a synchronous outer world. `scala.concurrent.Await` does that, with a timeout.

In [ ]:
import scala.concurrent.Await

val result: Int = Await.result(Future { 2 + 2 }, 1.second)
// 4 — blocks the current thread until the future completes (or throws on timeout)

Use `Await` only at the *outermost* boundary of your program — `main`, tests, a REPL session. Inside a service that handles many concurrent requests, blocking a thread defeats the purpose of using Futures at all. Worse, blocking on a Future from inside another Future running on the same thread pool can deadlock the pool entirely.

Treat `Await` as a smell whenever you see it in regular code. The fix is almost always to keep composing with `map`/`flatMap`/`for` until you reach the boundary.

## Common pitfalls

The mistakes you'll meet first, all of which trip up everyone once:

- **Side effects in `map`.** A `map` block runs on the EC's thread, not on yours. Mutating shared state inside a `map` invites the usual concurrency bugs — races, visibility issues. Either keep `map` pure or use `synchronized` / atomics deliberately.
- **Forgetting eagerness.** `val f = Future { compute() }` *starts* `compute` immediately. If you wanted laziness, `Future` is the wrong tool — see the note on effect systems below.
- **Sequentialising by accident.** The `for` body trap from above. If your latency adds up suspiciously, check whether you're starting Futures inside or outside the for.
- **Blocking inside an EC thread.** Long-running `Thread.sleep` or synchronous I/O blocks an EC thread. Either use `Future.blocking { ... }` to mark the section (which can grow the pool), or run blocking work on a dedicated pool.
- **Holding a Future and forgetting it.** A failed Future that nobody attaches to silently swallows the exception — unless you've installed an `UncaughtExceptionHandler` on the EC. Always handle failures somewhere downstream.

## A note on effect systems

`Future` is the standard-library answer to async. It works, and it's enough for a great deal of code. Two limitations bite once your async surface grows:

- **Eagerness.** You can't pass a `Future` around without it already executing. Caching, retries, and explicit-construction patterns all suffer.
- **No cancellation.** A `Future` can't be cancelled once started — work continues until completion even if no one is listening.

Two ecosystem libraries solve both: **`cats-effect`** (with `IO[A]`) and **ZIO** (with `ZIO[R, E, A]`). Both wrap descriptions of async computations as *values* that you compose freely, then "run" once at the edge of your program. Both add cancellation, structured concurrency, resource safety, and richer error channels. You can write enormous Scala codebases without them, but if you find yourself reinventing retry logic, cancellation flags, or careful manual resource cleanup around Futures, take the time to learn one.

For this notebook's purposes, `Future` is enough. The combinator shapes you've learned transfer to `IO` and `ZIO` almost line-for-line — only the runtime semantics differ.

## Putting it together — a parallel fan-out

A small but realistic example. Fetch three users in parallel, derive a balance for each, and combine into a summary — with a fallback for any individual failure.

In [ ]:
import scala.concurrent.{Future, Await}
import scala.concurrent.ExecutionContext.Implicits.global
import scala.concurrent.duration.*

def fetchUser(id: Long): Future[String] = Future { Thread.sleep(50); s"user#$id" }
def fetchBalance(name: String): Future[Int] = Future {
  Thread.sleep(50)
  if name.endsWith("3") then sys.error("flaky") else name.length * 100
}

def safeBalance(id: Long): Future[(String, Int)] =
  (for
    name    <- fetchUser(id)
    balance <- fetchBalance(name)
  yield (name, balance))
    .recover { case _: RuntimeException => (s"user#$id", 0) }

val summary: Future[String] =
  Future.traverse(List(1L, 2L, 3L))(safeBalance).map { pairs =>
    pairs.map((n, b) => s"$n -> $b").mkString(", ")
  }

// At the boundary — a test, a main, a REPL — block once to see it:
Await.result(summary, 5.seconds)
// "user#1 -> 600, user#2 -> 600, user#3 -> 0"

Trace through what just happened. `traverse` started three pipelines in parallel. Each pipeline ran `fetchUser` then `fetchBalance` sequentially via the `for`. `recover` caught the failure on user#3 and substituted a zero balance, so one bad call didn't poison the whole result. Finally, a single `map` shaped the list of pairs into a printable summary. The only blocking call was at the very edge — `Await.result` — which is the only place blocking belongs.

## What's next

Notebook 14 brings the threads together: **error handling and resource management**. How to model expected failures (`Either`, `Try`, custom ADTs) versus unexpected ones (exceptions), `try`/`finally` and the `using` resource pattern, and how the patterns extend into the async world you just met. After that, notebook 15 puts everything to work in the Scala-for-Spark setting that motivates this whole track.